In [5]:
# Syed Hussain Mehdi Raza
# 25K-9016
# Assignment 2
# Deep Learning
# YOLO

In [6]:
!pip -q install ultralytics pycocotools tqdm pyyaml pandas seaborn

import json
import shutil
import zipfile
from pathlib import Path
from typing import Dict, List, Tuple
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import yaml
import requests

from ultralytics import YOLO

plt.style.use('seaborn-v0_8')
np.random.seed(259016)

In [7]:
#  My ID is 9016
#  Last 2 digits are 16
#  Modulo 10 is 6
#  + 5
#  N is 11
N = 11

EPOCHS = 5 # Not having better GPU or paid google colab, so 5 epochs sir 🙏
IMG_SIZE = 640
BATCH = 16
MODEL_NAME = 'yolov8n.pt'

TRAINING_DATA_URL = 'http://images.cocodataset.org/zips/train2017.zip'
VALIDATION_DATA_URL = 'http://images.cocodataset.org/zips/val2017.zip'
ANNOTATIONS_DATA_URL = 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip'


ROOT = Path('/content')
DATA_ROOT = ROOT / 'datasets'
COCO_ROOT = DATA_ROOT / 'coco2017'
SUBSET_ROOT = DATA_ROOT / f'coco_subset_n{N}'
RUNS_ROOT = ROOT / 'runs'

print(f'N={N}')
print(f'COCO_ROOT={COCO_ROOT}')
print(f'SUBSET_ROOT={SUBSET_ROOT}')

N=11
COCO_ROOT=/content/datasets/coco2017
SUBSET_ROOT=/content/datasets/coco_subset_n11


In [8]:
# Download and extract dataset functions
# tqdm and ipywidgets for download progress

def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)

def download_file(url, filename: Path):
    """
    Downloads a file from a URL with a progress bar in a Jupyter Notebook.
    """
    ensure_dir(filename.parent)
    response = requests.get(url, stream=True, allow_redirects=True)
    if response.status_code != 200:
        raise RuntimeError(f"Request to {url} returned status code {response.status_code}")

    # Get the total file size from the response headers
    total_size = int(response.headers.get("content-length", 0))
    block_size = 1024 # 1 KB chunks

    # Use tqdm as a context manager to display the progress bar
    with tqdm(total=total_size, unit="B", unit_scale=True, desc=str(filename), leave=True) as progress_bar:
        with open(filename, "wb") as file:
            for data in response.iter_content(block_size):
                progress_bar.update(len(data))
                file.write(data)

    if total_size != 0 and progress_bar.n != total_size:
        print("Warning: Downloaded file size does not match expected size")

def extract_zip(zip_path: Path, target_dir: Path, expected_path: Path = None) -> None:
    if expected_path is not None and expected_path.exists():
        print(f'[skip] already extracted: {expected_path}')
        return

    print(f'[extract] {zip_path.name} -> {target_dir}')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        ensure_dir(target_dir)
        members = zf.infolist()
        with tqdm(total=len(members), unit="file", desc=str(zip_path.name), leave=True) as progress_bar:
            for member in members:
                zf.extract(member, target_dir)
                progress_bar.update(1)
    print('[done] extraction complete')

In [9]:
# Downloading start, step by step
#  Annotations first small zip
annotations_zip = COCO_ROOT / 'annotations_trainval2017.zip'
download_file(ANNOTATIONS_DATA_URL, annotations_zip);

/content/datasets/coco2017/annotations_trainval2017.zip:   0%|          | 0.00/253M [00:00<?, ?B/s]

In [10]:
# Download Training Data
train_zip = COCO_ROOT / 'train2017.zip'
download_file(TRAINING_DATA_URL, train_zip);

/content/datasets/coco2017/train2017.zip:   0%|          | 0.00/19.3G [00:00<?, ?B/s]

In [11]:
# Download validation Data
val_zip = COCO_ROOT / 'val2017.zip'
download_file(VALIDATION_DATA_URL, val_zip);

/content/datasets/coco2017/val2017.zip:   0%|          | 0.00/816M [00:00<?, ?B/s]

In [12]:
# Extraction Time
extract_zip(annotations_zip, COCO_ROOT)

[extract] annotations_trainval2017.zip -> /content/datasets/coco2017


annotations_trainval2017.zip:   0%|          | 0/6 [00:00<?, ?file/s]

[done] extraction complete


In [13]:
extract_zip(train_zip, COCO_ROOT)

[extract] train2017.zip -> /content/datasets/coco2017


train2017.zip:   0%|          | 0/118288 [00:00<?, ?file/s]

[done] extraction complete


In [14]:
# Select first N COCO classes and build YOLO train/val/test subset in-notebook

from typing import Dict, List, Tuple


def load_coco_json(path: Path) -> Dict:
    with open(path, 'r') as f:
        return json.load(f)


def select_first_n_category_ids(categories: List[Dict], n: int) -> Tuple[List[int], List[str]]:
    """
    First N classes = categories sorted by COCO category id, taking first n.
    """
    sorted_cats = sorted(categories, key=lambda c: int(c['id']))
    if n <= 0:
        raise ValueError('N must be > 0')
    if n > len(sorted_cats):
        raise ValueError(f'N={n} exceeds available categories={len(sorted_cats)}')

    selected = sorted_cats[:n]
    selected_ids = [int(c['id']) for c in selected]
    selected_names = [str(c['name']) for c in selected]
    return selected_ids, selected_names


def coco_bbox_to_yolo(bbox: List[float], img_w: int, img_h: int) -> Tuple[float, float, float, float]:
    x, y, w, h = bbox
    x_center = (x + w / 2.0) / img_w
    y_center = (y + h / 2.0) / img_h
    w_norm = w / img_w
    h_norm = h / img_h
    return x_center, y_center, w_norm, h_norm


def safe_link_or_copy(src: Path, dst: Path) -> None:
    if dst.exists():
        return
    try:
        dst.symlink_to(src)
    except Exception:
        shutil.copy2(src, dst)


def build_yolo_subset_split(
    coco_data: Dict,
    image_root: Path,
    split_name: str,
    subset_root: Path,
    selected_cat_ids: List[int],
    class_id_to_index: Dict[int, int],
    allowed_image_ids: set = None,
) -> Dict[str, int]:
    img_meta = {img['id']: img for img in coco_data['images']}
    anns_by_img: Dict[int, List[Dict]] = {}

    selected_set = set(selected_cat_ids)

    for ann in coco_data['annotations']:
        if ann.get('iscrowd', 0) == 1:
            continue
        if ann['category_id'] not in selected_set:
            continue
        if ann['bbox'][2] <= 1 or ann['bbox'][3] <= 1:
            continue

        image_id = ann['image_id']
        if allowed_image_ids is not None and image_id not in allowed_image_ids:
            continue

        anns_by_img.setdefault(image_id, []).append(ann)

    split_img_dir = subset_root / 'images' / split_name
    split_lbl_dir = subset_root / 'labels' / split_name
    ensure_dir(split_img_dir)
    ensure_dir(split_lbl_dir)

    kept_images = 0
    kept_labels = 0

    for image_id, anns in tqdm(anns_by_img.items(), total=len(anns_by_img), desc=f'Preparing {split_name}'):
        meta = img_meta.get(image_id)
        if meta is None:
            continue

        src_img = image_root / meta['file_name']
        if not src_img.exists():
            continue

        dst_img = split_img_dir / meta['file_name']
        safe_link_or_copy(src_img, dst_img)

        label_path = split_lbl_dir / f"{Path(meta['file_name']).stem}.txt"
        lines = []
        for ann in anns:
            cls_idx = class_id_to_index[ann['category_id']]
            x_c, y_c, w_n, h_n = coco_bbox_to_yolo(ann['bbox'], meta['width'], meta['height'])
            lines.append(f"{cls_idx} {x_c:.6f} {y_c:.6f} {w_n:.6f} {h_n:.6f}")
            kept_labels += 1

        with open(label_path, 'w') as f:
            f.write('\n'.join(lines) + ('\n' if lines else ''))

        kept_images += 1

    return {'images': kept_images, 'labels': kept_labels}


# ---- Configure split behavior ----
TEST_RATIO_FROM_VAL = 0.20  # 20% of val images become test set

train_json = COCO_ROOT / 'annotations' / 'instances_train2017.json'
val_json = COCO_ROOT / 'annotations' / 'instances_val2017.json'
assert train_json.exists(), f'Missing: {train_json}'
assert val_json.exists(), f'Missing: {val_json}'

train_data = load_coco_json(train_json)
val_data = load_coco_json(val_json)

selected_cat_ids, selected_names = select_first_n_category_ids(train_data['categories'], N)
class_id_to_index = {cid: idx for idx, cid in enumerate(selected_cat_ids)}

print('Selected first N COCO classes (by category id):')
for idx, (cid, name) in enumerate(zip(selected_cat_ids, selected_names)):
    print(f'  {idx:2d}: coco_id={cid:2d}, name={name}')

# Train split from train2017
ensure_dir(SUBSET_ROOT)
stats_train = build_yolo_subset_split(
    coco_data=train_data,
    image_root=COCO_ROOT / 'train2017',
    split_name='train2017',
    subset_root=SUBSET_ROOT,
    selected_cat_ids=selected_cat_ids,
    class_id_to_index=class_id_to_index,
)

# Val/Test split from val2017 (deterministic by sorted image ids)
val_img_ids_sorted = sorted([img['id'] for img in val_data['images']])
num_test = int(len(val_img_ids_sorted) * TEST_RATIO_FROM_VAL)
test_ids = set(val_img_ids_sorted[:num_test])
val_ids = set(val_img_ids_sorted[num_test:])

stats_val = build_yolo_subset_split(
    coco_data=val_data,
    image_root=COCO_ROOT / 'val2017',
    split_name='val2017',
    subset_root=SUBSET_ROOT,
    selected_cat_ids=selected_cat_ids,
    class_id_to_index=class_id_to_index,
    allowed_image_ids=val_ids,
)

stats_test = build_yolo_subset_split(
    coco_data=val_data,
    image_root=COCO_ROOT / 'val2017',
    split_name='test2017',
    subset_root=SUBSET_ROOT,
    selected_cat_ids=selected_cat_ids,
    class_id_to_index=class_id_to_index,
    allowed_image_ids=test_ids,
)

# YOLO dataset yaml
DATA_YAML = SUBSET_ROOT / 'coco_subset.yaml'
yaml_dict = {
    'path': str(SUBSET_ROOT),
    'train': 'images/train2017',
    'val': 'images/val2017',
    'test': 'images/test2017',
    'names': {i: name for i, name in enumerate(selected_names)},
}

with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(yaml_dict, f, sort_keys=False)

print('\nSubset build complete:')
print(f"  train images={stats_train['images']}, labels={stats_train['labels']}")
print(f"  val images={stats_val['images']}, labels={stats_val['labels']}")
print(f"  test images={stats_test['images']}, labels={stats_test['labels']}")
print(f'Wrote: {DATA_YAML}')
print(yaml.safe_dump(yaml_dict, sort_keys=False))

Selected first N COCO classes (by category id):
   0: coco_id= 1, name=person
   1: coco_id= 2, name=bicycle
   2: coco_id= 3, name=car
   3: coco_id= 4, name=motorcycle
   4: coco_id= 5, name=airplane
   5: coco_id= 6, name=bus
   6: coco_id= 7, name=train
   7: coco_id= 8, name=truck
   8: coco_id= 9, name=boat
   9: coco_id=10, name=traffic light
  10: coco_id=11, name=fire hydrant


Preparing train2017:   0%|          | 0/75386 [00:00<?, ?it/s]

Preparing val2017:   0%|          | 0/2541 [00:00<?, ?it/s]

Preparing test2017:   0%|          | 0/634 [00:00<?, ?it/s]


Subset build complete:
  train images=75386, labels=367502
  val images=0, labels=0
  test images=0, labels=0
Wrote: /content/datasets/coco_subset_n11/coco_subset.yaml
path: /content/datasets/coco_subset_n11
train: images/train2017
val: images/val2017
test: images/test2017
names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: airplane
  5: bus
  6: train
  7: truck
  8: boat
  9: traffic light
  10: fire hydrant



In [15]:
# Start training
assert DATA_YAML.exists(), f'Dataset YAML not found: {DATA_YAML}'

model = YOLO(MODEL_NAME)
run_name = f'yolo_coco_subset_n{N}'

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    project=str(RUNS_ROOT),
    name=run_name,
    exist_ok=True,
    workers=2,
)

print('Training completed.')

best_model_path = RUNS_ROOT / run_name / 'weights' / 'best.pt'
last_model_path = RUNS_ROOT / run_name / 'weights' / 'last.pt'
print(f'Best model: {best_model_path}')
print(f'Last model: {last_model_path}')

# Quick validation on val split
best_model = YOLO(str(best_model_path if best_model_path.exists() else last_model_path))
metrics = best_model.val(data=str(DATA_YAML), split='val', imgsz=IMG_SIZE)

print('\nValidation metrics:')
print(f"  mAP@0.5       : {float(metrics.box.map50):.4f}")
print(f"  mAP@0.5:0.95  : {float(metrics.box.map):.4f}")
print(f"  precision     : {float(metrics.box.mp):.4f}")
print(f"  recall        : {float(metrics.box.mr):.4f}")

Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/coco_subset_n11/coco_subset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_coco_subset_n11, nbs=64, nms=False, opset=None, optimize=False, optimizer=aut

FileNotFoundError: [34m[1mval: [0mError loading data from /content/datasets/coco_subset_n11/images/val2017
See https://docs.ultralytics.com/datasets for dataset formatting guidance.